# Limpieza de datos: libros más vendidos de Amazon (2009-2019)

**Objetivo:** estandarizar columnas, validar valores sospechosos y generar un dataset limpio en `data/processed/`.

In [1]:
# Importar la librería principal para análisis de datos
import pandas as pd

# Mostrar todas las columnas al imprimir tablas (evita los "...")
pd.set_option("display.max_columns", None)

# Cargar el CSV original desde data/raw
print("=== CARGA DE DATOS ===")
ruta = "../data/raw/amazon_bestsellers_2009-2019.csv"
df_raw = pd.read_csv(ruta)

# Trabajar sobre una copia para no modificar el original
df = df_raw.copy()

print("Dimensiones del dataset original (filas, columnas):", df_raw.shape)

=== CARGA DE DATOS ===
Dimensiones del dataset original (filas, columnas): (550, 7)


In [2]:
# Convertir nombres de columnas a snake_case
print("=== CONVERSIÓN A SNAKE_CASE ===")
print("Nombres antes de la conversión:")
print(list(df.columns))

# Quitar espacios en los bordes, pasar a minúsculas y cambiar espacios por "_"
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nNombres después de la conversión:")
print(list(df.columns))

=== CONVERSIÓN A SNAKE_CASE ===
Nombres antes de la conversión:
['Name', 'Author', 'User Rating', 'Reviews', 'Price', 'Year', 'Genre']

Nombres después de la conversión:
['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre']


In [3]:
# Limpiar espacios sobrantes en columnas de texto
print("=== LIMPIEZA DE TEXTO ===")

# Relación entre el nombre nuevo de la columna y el original (para comparar)
columnas_texto = {"name": "Name", "author": "Author", "genre": "Genre"}

for col_nueva, col_original in columnas_texto.items():
    # Quitar espacios al inicio/final y reemplazar espacios múltiples por uno solo
    df[col_nueva] = (
        df[col_nueva]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Contar cuántos valores cambiaron respecto al original
    cambios = (df_raw[col_original] != df[col_nueva]).sum()
    print(f"Columna '{col_nueva}': {cambios} valores modificados")

# Comparar títulos únicos antes y después de la limpieza
print("\nTítulos únicos antes:", df_raw["Name"].nunique())
print("Títulos únicos después:", df["name"].nunique())

=== LIMPIEZA DE TEXTO ===
Columna 'name': 0 valores modificados
Columna 'author': 2 valores modificados
Columna 'genre': 0 valores modificados

Títulos únicos antes: 351
Títulos únicos después: 351


In [4]:
# Investigar precios sospechosos (mínimo = 0 y máximo = 105)
print("=== ANÁLISIS DE PRECIOS SOSPECHOSOS ===")

# Libros con precio 0
print("\n--- Libros con precio 0 ---")
precio_cero = df[df["price"] == 0][["name", "author", "year", "price"]]
print("Cantidad de libros con precio 0:", len(precio_cero))
display(precio_cero)

# Los 5 precios más altos
print("\n--- Los 5 precios más altos ---")
display(df.nlargest(5, "price")[["name", "author", "year", "price"]])

# Crear una columna que marque los libros gratuitos (en vez de borrarlos)
df["is_free"] = df["price"] == 0
print("\nColumna 'is_free' creada. Libros gratuitos:", df["is_free"].sum())

=== ANÁLISIS DE PRECIOS SOSPECHOSOS ===

--- Libros con precio 0 ---
Cantidad de libros con precio 0: 12


,name,author,year,price
42,"Cabin Fever (Diary of a Wimpy Kid, Book 6)",Jeff Kinney,2011,0
71,"Diary of a Wimpy Kid: Hard Luck, Book 8",Jeff Kinney,2013,0
116,Frozen (Little Golden Book),RH Disney,2014,0
193,JOURNEY TO THE ICE P,RH Disney,2014,0
219,Little Blue Truck,Alice Schertle,2014,0
358,The Constitution of the United States,Delegates of the Constitutional,2016,0
381,The Getaway,Jeff Kinney,2017,0
461,The Short Second Life of Bree Tanner: An Eclip...,Stephenie Meyer,2010,0
505,To Kill a Mockingbird,Harper Lee,2013,0
506,To Kill a Mockingbird,Harper Lee,2014,0



--- Los 5 precios más altos ---


,name,author,year,price
69,Diagnostic and Statistical Manual of Mental Di...,American Psychiatric Association,2013,105
70,Diagnostic and Statistical Manual of Mental Di...,American Psychiatric Association,2014,105
473,The Twilight Saga Collection,Stephenie Meyer,2009,82
151,Hamilton: The Revolution,Lin-Manuel Miranda,2016,54
346,The Book of Basketball: The NBA According to T...,Bill Simmons,2009,53



Columna 'is_free' creada. Libros gratuitos: 12


In [5]:
# Verificar que las repeticiones de títulos entre años son legítimas
print("=== CONSISTENCIA DE TÍTULOS REPETIDOS ===")

# ¿Hay un mismo título más de una vez en el mismo año? (no debería)
repetidos_mismo_anio = df.duplicated(subset=["name", "year"]).sum()
print("Títulos repetidos en el mismo año:", repetidos_mismo_anio)

# ¿Algún título tiene más de un autor distinto?
autores_por_titulo = df.groupby("name")["author"].nunique()
titulos_multi_autor = autores_por_titulo[autores_por_titulo > 1]
print("Títulos con más de un autor:", len(titulos_multi_autor))

# ¿Algún título cambia de género entre años?
generos_por_titulo = df.groupby("name")["genre"].nunique()
titulos_multi_genero = generos_por_titulo[generos_por_titulo > 1]
print("Títulos con más de un género:", len(titulos_multi_genero))

# Si hay inconsistencias, mostrar las filas afectadas para revisarlas
if len(titulos_multi_autor) > 0:
    print("\n--- Títulos con más de un autor ---")
    display(df[df["name"].isin(titulos_multi_autor.index)].sort_values("name"))

if len(titulos_multi_genero) > 0:
    print("\n--- Títulos con más de un género ---")
    display(df[df["name"].isin(titulos_multi_genero.index)].sort_values("name"))

# Libros que más veces aparecen en el top 50
print("\n--- Los 10 libros que más aparecen en el top 50 ---")
display(df["name"].value_counts().head(10))

=== CONSISTENCIA DE TÍTULOS REPETIDOS ===
Títulos repetidos en el mismo año: 3
Títulos con más de un autor: 0
Títulos con más de un género: 0

--- Los 10 libros que más aparecen en el top 50 ---


name
Publication Manual of the American Psychological Association, 6th Edition            10
StrengthsFinder 2.0                                                                   9
Oh, the Places You'll Go!                                                             8
The 7 Habits of Highly Effective People: Powerful Lessons in Personal Change          7
The Very Hungry Caterpillar                                                           7
Jesus Calling: Enjoying Peace in His Presence (with Scripture References)             6
The Four Agreements: A Practical Guide to Personal Freedom (A Toltec Wisdom Book)     6
First 100 Words                                                                       5
Giraffes Can't Dance                                                                  5
How to Win Friends & Influence People                                                 5
Name: count, dtype: int64

In [6]:
# Validaciones automáticas de calidad de datos
# (si alguna falla, se muestra un error con el mensaje indicado)
print("=== VALIDACIONES DE CALIDAD ===")

assert df["user_rating"].between(1, 5).all(), "Hay valoraciones fuera del rango 1-5"
assert (df["reviews"] >= 0).all(), "Hay reseñas negativas"
assert (df["price"] >= 0).all(), "Hay precios negativos"
assert df["year"].between(2009, 2019).all(), "Hay años fuera del rango 2009-2019"
assert df.isnull().sum().sum() == 0, "Hay valores nulos"

print(" Todas las validaciones pasaron")

=== VALIDACIONES DE CALIDAD ===
 Todas las validaciones pasaron


In [7]:
# Investigar los títulos repetidos dentro del mismo año
print("=== REVISIÓN DE TÍTULOS REPETIDOS EN EL MISMO AÑO ===")

# keep=False marca TODAS las filas involucradas, no solo la segunda
repetidos = df[df.duplicated(subset=["name", "year"], keep=False)]
display(repetidos.sort_values(["name", "year"]))

# Verificar cuántas filas tiene cada año (si el top 50 está completo, deben ser 50)
print("\n--- Filas por año ---")
print(df.groupby("year").size())

=== REVISIÓN DE TÍTULOS REPETIDOS EN EL MISMO AÑO ===


,name,author,user_rating,reviews,price,year,genre,is_free
367,The Fault in Our Stars,John Green,4.7,50482,7,2014,Fiction,False
368,The Fault in Our Stars,John Green,4.7,50482,13,2014,Fiction,False
404,The Help,Kathryn Stockett,4.8,13871,8,2011,Fiction,False
405,The Help,Kathryn Stockett,4.8,13871,7,2011,Fiction,False
518,"Unbroken: A World War II Story of Survival, Re...",Laura Hillenbrand,4.8,29673,13,2014,Non Fiction,False
519,"Unbroken: A World War II Story of Survival, Re...",Laura Hillenbrand,4.8,29673,16,2014,Non Fiction,False



--- Filas por año ---
year
2009    50
2010    50
2011    50
2012    50
2013    50
2014    50
2015    50
2016    50
2017    50
2018    50
2019    50
dtype: int64


In [8]:
# Guardar el dataset limpio en data/processed
print("=== GUARDADO DEL DATASET LIMPIO ===")

ruta_salida = "../data/processed/amazon_bestsellers_clean.csv"

# index=False evita guardar el número de fila como una columna extra
df.to_csv(ruta_salida, index=False)

print("Archivo guardado en:", ruta_salida)
print("Dimensiones finales (filas, columnas):", df.shape)
print("Columnas finales:", list(df.columns))

=== GUARDADO DEL DATASET LIMPIO ===
Archivo guardado en: ../data/processed/amazon_bestsellers_clean.csv
Dimensiones finales (filas, columnas): (550, 8)
Columnas finales: ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre', 'is_free']


# Limpieza de datos: libros más vendidos de Amazon (2009-2019)

**Objetivo:** preparar el dataset para el análisis posterior, manteniendo la información útil y eliminando inconsistencias que pudieran distorsionar los resultados.

Tras la exploración inicial, comprobamos que el conjunto tenía 550 filas y 7 columnas, con una estructura bastante ordenada. La limpieza no buscó “corregir” todo a la fuerza, sino definir reglas coherentes para normalizar los datos sin perder contexto analítico.

## Decisiones clave de limpieza

### 1. Estandarización de nombres de columnas
Se renombraron las columnas para que siguieran un formato consistente y más cómodo para trabajar en pandas:

- `Name` → `name`
- `Author` → `author`
- `User Rating` → `user_rating`
- `Reviews` → `reviews`
- `Price` → `price`
- `Year` → `year`
- `Genre` → `genre`

Esto se hizo usando `snake_case`, lo que simplifica el acceso a las columnas y evita errores por espacios o mayúsculas inconsistentes.

### 2. Normalización de texto
Se eliminaron espacios sobrantes al inicio y al final de cada texto, y también se regularizaron espacios duplicados dentro de cadenas. Esta limpieza fue especialmente útil para:

- nombres de libros
- autores
- géneros

La limpieza modificó 2 valores de texto en total: ambos en `author` (0 valores en `name` y 0 en `genre`). Además, la cantidad de títulos únicos se mantuvo en 351 antes y después de normalizar.

### 3. Tratamiento de precios sospechosos
Se encontraron 12 libros con `price = 0`. En lugar de eliminarlos automáticamente, se creó una variable auxiliar:

- `is_free = True` cuando el precio es 0

Un caso llamativo es `To Kill a Mockingbird`, que aparece con precio 0 durante cuatro años consecutivos (2013-2016), es decir, 4 de los 12 registros gratuitos. Esto sugiere que podría tratarse de una edición gratuita recurrente. En general, un precio cero puede corresponder a un libro gratuito, una promoción o un caso especial; por eso se preservaron los registros y se dejó abierta la posibilidad de excluirlos solo en análisis específicos.

También se revisaron los precios altos y no se consideraron errores: el `Diagnostic and Statistical Manual of Mental Disorders (DSM-5)` aparece a 105 y `The Twilight Saga Collection` a 82. Son casos plausibles por tratarse de un libro de referencia y un box set, respectivamente.

### 4. Verificación de duplicados y consistencia
Se revisó si las repeticiones eran reales o indicaban problemas en el dato:

- no había títulos con más de un autor distintos
- no había títulos con más de un género distintos
- los títulos repetidos en diferentes años se consideraron válidos, ya que la base recoge rankings anuales y un libro puede volver a aparecer en varios años

La comprobación principal fue si el mismo título aparecía más de una vez en el mismo año. Se encontraron 3 títulos repetidos: `The Fault in Our Stars` (2014), `The Help` (2011) y `Unbroken: A World War II Story of Survival, Resilience, and Redemption` (2014). Se conservaron las filas porque tienen precios distintos y la revisión no justificó eliminarlas automáticamente.

### 5. Validación de rangos y nulos
Se aplicaron validaciones de calidad para comprobar que el dataset estaba listo para análisis:

- `user_rating` en el rango 1 a 5
- `reviews` no negativas
- `price` no negativas
- `year` dentro del rango 2009-2019
- ausencia de valores nulos

Estas comprobaciones actúan como una “salida de seguridad”: si se introdujera un dato fuera de rango, el análisis no seguiría avanzando con información inválida.

### 6. Limitaciones de la fuente
Se observaron textos truncados en los datos originales, como el título `JOURNEY TO THE ICE P` y el autor `Delegates of the Constitutional`. Se dejaron sin corregir porque no es posible reconstruir con certeza el texto faltante a partir de este dataset; esto limita la interpretación y el posible emparejamiento de esos registros.

### 7. Guardado del dataset limpio
Una vez finalizada la limpieza, se exportó el resultado a la carpeta de datos procesados, manteniendo la estructura original pero con columnas uniformes y validaciones aplicadas.

## Resultado
El dataset quedó listo para el análisis posterior, manteniendo la información relevante, evitando ruido por formato y distinguiendo entre errores reales, limitaciones de la fuente y casos legítimos dentro de la lista de bestsellers.

En resumen, la estrategia fue:
- normalizar formato,
- preservar información útil,
- marcar anomalías cuando correspondía,
- y validar la consistencia antes de guardar el resultado final.